# Latent Diffusion Models: The Magic Behind Stable Diffusion

**Learning Objectives:**
- Understand why pixel-space diffusion is computationally expensive
- Learn how VAE + Diffusion enables 8x compression and 64x speed improvement
- Implement a complete latent diffusion model from scratch
- Master text conditioning via cross-attention mechanisms
- Implement classifier-free guidance for better generation quality
- Understand the complete Stable Diffusion architecture
- Apply latent diffusion to text-to-image generation

**What We'll Build:**
1. A VAE for perceptual compression (image → latent → image)
2. A diffusion model that operates in latent space (not pixel space!)
3. A U-Net with cross-attention for text conditioning
4. A complete text-to-image pipeline on MNIST
5. Classifier-free guidance for quality improvements
6. DDIM sampling for fast generation

## Part 1: Introduction - Why Latent Diffusion?

### The Problem with Pixel-Space Diffusion

You've learned how **Denoising Diffusion Probabilistic Models (DDPMs)** work by gradually adding and removing noise. But there's a critical problem:

**Pixel-space diffusion is SLOW:**
- For a 512×512 RGB image: 786,432 pixels to denoise!
- U-Net processes every pixel at every timestep
- 1000 denoising steps × massive resolution = prohibitive compute
- Training requires enormous GPU memory and time

**Example:**
- 512×512 image: ~260k pixels
- 50 DDIM steps
- Large U-Net: ~1 billion parameters
- Result: 5-10 seconds per image on high-end GPU

This makes training and inference impractical for high-resolution images!

### The Breakthrough: Latent Diffusion

**Key insight from "High-Resolution Image Synthesis with Latent Diffusion Models" (Rombach et al., 2022):**

Instead of denoising in pixel space, **denoise in a compressed latent space**!

**The architecture:**
```
Training:
Image (512×512×3) → [VAE Encoder] → Latent (64×64×4) → [Add Noise] → Noisy Latent
                                                           ↓
                                         [U-Net predicts noise] ← Text Embedding
                                                           ↓
Denoised Latent → [VAE Decoder] → Reconstructed Image (512×512×3)

Sampling:
Random Noise (64×64×4) → [Iterative Denoising with U-Net] → Clean Latent → [VAE Decoder] → Image
```

### The Improvements

**1. Compression (8× spatial reduction):**
- 512×512 image → 64×64 latent
- Reduces dimensionality by factor of 64 (8×8)
- From 262,144 pixels to 4,096 latent elements

**2. Speed (up to 64× faster):**
- U-Net operates on 64×64 instead of 512×512
- 64× fewer elements to process
- Much faster training and sampling

**3. Perceptual compression:**
- Latents capture semantic information ("what" is in the image)
- Removes high-frequency details that don't affect perception
- VAE decoder reconstructs perceptually important details

**4. Text conditioning:**
- Add cross-attention layers in U-Net
- Text embeddings guide denoising process
- Enables text-to-image generation!

### Real-World Impact

This architecture directly powers or extends:
- **Stable Diffusion** (most popular open-source latent diffusion model)
- **ControlNet** (structure-guided control built on latent diffusion)
- Many open-source text-to-image and image-editing systems derived from Stable Diffusion

Understanding latent diffusion is essential for modern generative AI!

## Part 2: Setup and Imports

Let's import everything we need and configure our environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math

# Import from shared library
from aiml_notebooks import get_device, set_seed

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device(prefer_cpu=False)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Part 3: Load MNIST Dataset

We'll use MNIST to demonstrate latent diffusion concepts quickly. The same principles apply to high-resolution images like in Stable Diffusion!

In [ ]:
# Transform: normalize to [-1, 1] (standard for diffusion models)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Maps [0, 1] to [-1, 1]
])

# Load datasets
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./tmp/data',
    train=False,
    download=True,
    transform=transform
)

# Create dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Image shape: {train_dataset[0][0].shape}")  # (1, 28, 28)

## Part 4: Review - Pixel-Space DDPM Recap

Let's quickly review how standard DDPMs work before moving to latent space.

### Forward Process (Adding Noise)

Given a clean image $x_0$, we can add noise to get $x_t$ at timestep $t$:

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon$$

where $\epsilon \sim \mathcal{N}(0, I)$ is random Gaussian noise.

### Reverse Process (Denoising)

Train a U-Net $\epsilon_\theta(x_t, t)$ to predict the noise:

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

### Why This is Expensive

**For MNIST (28×28):**
- 784 pixels per image
- Not too bad!

**For high-res images (512×512×3):**
- 786,432 pixels per image
- U-Net must process every pixel
- Memory usage: batch_size × channels × height × width × multiple feature maps
- Result: SLOW training and inference

### The Solution: Work in Latent Space

**Instead of:**
```
Image (512×512×3) → [Diffusion] → Image
```

**Do this:**
```
Image → [VAE Encode] → Latent (64×64×4) → [Diffusion] → Latent → [VAE Decode] → Image
```

The diffusion happens in a much smaller latent space!

## Part 5: Building the VAE Component

### Why VAE for Compression?

We need a **perceptual compressor** that:
1. Compresses images to smaller representations (encoder)
2. Reconstructs high-quality images from latents (decoder)
3. Preserves semantic information while removing redundancy

A **Variational Autoencoder (VAE)** is perfect because:
- Learns smooth, continuous latent space
- Compresses by removing perceptually unimportant details
- Decoder can reconstruct high-quality images
- Regularized latent space (via KL divergence)

### VAE Architecture for MNIST

For MNIST (28×28), we'll compress to a smaller spatial resolution:
- **Input**: 28×28 image
- **Latent**: 7×7×4 (196 elements, 4× compression)
- **Output**: 28×28 reconstructed image

This demonstrates the concept while keeping training fast!

## Part 6: Implementing the VAE Encoder

The encoder compresses images into a latent representation using convolutional layers.

In [ ]:
class VAEEncoder(nn.Module):
    """
    VAE Encoder: Image → Latent distribution
    
    Compresses 28×28 image to 7×7 latent space.
    Outputs mean and log-variance for reparameterization trick.
    """
    def __init__(self, in_channels=1, latent_channels=4, hidden_dims=[32, 64]):
        super().__init__()
        
        # Convolutional encoder
        modules = []
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, h_dim, kernel_size=3, stride=2, padding=1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU(0.2)
                )
            )
            in_channels = h_dim
        
        self.encoder = nn.Sequential(*modules)
        
        # Output: mean and log-variance for latent distribution
        self.fc_mu = nn.Conv2d(hidden_dims[-1], latent_channels, kernel_size=3, padding=1)
        self.fc_logvar = nn.Conv2d(hidden_dims[-1], latent_channels, kernel_size=3, padding=1)
    
    def forward(self, x):
        """
        Args:
            x: Images (batch_size, 1, 28, 28)
        
        Returns:
            mu: Mean of latent distribution (batch_size, latent_channels, 7, 7)
            logvar: Log-variance of latent distribution
        """
        h = self.encoder(x)  # (batch, hidden_dims[-1], 7, 7)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

# Test encoder
encoder = VAEEncoder(in_channels=1, latent_channels=4).to(device)
test_img = torch.randn(4, 1, 28, 28).to(device)
test_mu, test_logvar = encoder(test_img)

print(f"Encoder test:")
print(f"  Input shape: {test_img.shape}")
print(f"  Mu shape: {test_mu.shape}")
print(f"  Logvar shape: {test_logvar.shape}")
print(f"  Compression: {test_img.numel() / test_mu.numel():.1f}× spatial reduction")

## Part 7: Implementing the VAE Decoder

The decoder reconstructs images from latent representations using transposed convolutions.

In [ ]:
class VAEDecoder(nn.Module):
    """
    VAE Decoder: Latent → Image
    
    Reconstructs 28×28 image from 7×7 latent space.
    """
    def __init__(self, latent_channels=4, out_channels=1, hidden_dims=[64, 32]):
        super().__init__()
        
        # Initial projection
        self.decoder_input = nn.Conv2d(latent_channels, hidden_dims[0], kernel_size=3, padding=1)
        
        # Transposed convolutions for upsampling
        modules = []
        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(hidden_dims[i], hidden_dims[i+1], 
                                     kernel_size=3, stride=2, padding=1, output_padding=1),
                    nn.BatchNorm2d(hidden_dims[i+1]),
                    nn.LeakyReLU(0.2)
                )
            )
        
        self.decoder = nn.Sequential(*modules)
        
        # Final layer to output image
        self.final_layer = nn.Sequential(
            nn.ConvTranspose2d(hidden_dims[-1], out_channels,
                             kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z):
        """
        Args:
            z: Latent code (batch_size, latent_channels, 7, 7)
        
        Returns:
            Reconstructed image (batch_size, 1, 28, 28)
        """
        h = self.decoder_input(z)  # (batch, hidden_dims[0], 7, 7)
        h = self.decoder(h)  # (batch, hidden_dims[-1], 14, 14)
        return self.final_layer(h)  # (batch, 1, 28, 28)

# Test decoder
decoder = VAEDecoder(latent_channels=4, out_channels=1).to(device)
test_z = torch.randn(4, 4, 7, 7).to(device)
test_recon = decoder(test_z)

print(f"Decoder test:")
print(f"  Input shape: {test_z.shape}")
print(f"  Output shape: {test_recon.shape}")
print(f"  Output range: [{test_recon.min():.2f}, {test_recon.max():.2f}]")

## Part 8: Complete VAE with Reparameterization Trick

We combine the encoder and decoder into a complete VAE. The **reparameterization trick** allows us to sample from the latent distribution while maintaining gradient flow.

In [ ]:
class VAE(nn.Module):
    """
    Complete VAE for perceptual compression.
    
    This will be the 'compression' component of latent diffusion.
    """
    def __init__(self, in_channels=1, latent_channels=4):
        super().__init__()
        
        self.encoder = VAEEncoder(in_channels, latent_channels)
        self.decoder = VAEDecoder(latent_channels, in_channels)
        self.latent_channels = latent_channels
    
    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + sigma * epsilon
        
        This allows backpropagation through sampling.
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def encode(self, x):
        """Encode image to latent distribution."""
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar
    
    def decode(self, z):
        """Decode latent to image."""
        return self.decoder(z)
    
    def forward(self, x):
        """Full forward pass."""
        z, mu, logvar = self.encode(x)
        recon = self.decode(z)
        return recon, mu, logvar, z

# Create VAE
vae = VAE(in_channels=1, latent_channels=4).to(device)

# Test
test_img = torch.randn(4, 1, 28, 28).to(device)
test_recon, test_mu, test_logvar, test_z = vae(test_img)

print(f"VAE test:")
print(f"  Input: {test_img.shape}")
print(f"  Latent: {test_z.shape}")
print(f"  Reconstruction: {test_recon.shape}")
print(f"  Total parameters: {sum(p.numel() for p in vae.parameters()):,}")

## Part 9: VAE Loss Function

The VAE loss combines two terms:

$$\mathcal{L}_{VAE} = \mathcal{L}_{recon} + \beta \cdot \mathcal{L}_{KL}$$

**1. Reconstruction loss** - How well we reconstruct the input:
$$\mathcal{L}_{recon} = \mathbb{E}_{q(z|x)}[\|x - \hat{x}\|^2]$$

**2. KL divergence** - How close the latent distribution is to standard normal:
$$\mathcal{L}_{KL} = D_{KL}(q(z|x) \| p(z)) = -\frac{1}{2}\sum(1 + \log(\sigma^2) - \mu^2 - \sigma^2)$$

The KL term regularizes the latent space, ensuring it's smooth and continuous.

In [ ]:
def vae_loss(recon, x, mu, logvar, beta=0.0001):
    """
    Compute VAE loss.
    
    Args:
        recon: Reconstructed images
        x: Original images
        mu: Mean of latent distribution
        logvar: Log-variance of latent distribution
        beta: Weight for KL term (lower = more emphasis on reconstruction)
    
    Returns:
        Total loss, reconstruction loss, KL loss
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon, x, reduction='sum') / x.size(0)
    
    # KL divergence loss
    # KL(q(z|x) || N(0,1)) = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    
    # Total loss
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss

# Test loss
test_img = torch.randn(4, 1, 28, 28).to(device)
test_recon, test_mu, test_logvar, _ = vae(test_img)
test_total, test_recon_loss, test_kl = vae_loss(test_recon, test_img, test_mu, test_logvar)

print(f"VAE loss test:")
print(f"  Total loss: {test_total.item():.4f}")
print(f"  Reconstruction: {test_recon_loss.item():.4f}")
print(f"  KL divergence: {test_kl.item():.4f}")

## Part 10: Training the VAE

Let's train the VAE to learn a good perceptual compression of MNIST digits. This will be our "compression engine" for latent diffusion!

In [ ]:
# Training configuration
vae_epochs = 10
vae_lr = 1e-3
vae_beta = 0.0001  # Low beta to prioritize reconstruction quality

# Optimizer
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=vae_lr)

# Training history
vae_history = {'total': [], 'recon': [], 'kl': []}

print(f"Training VAE for {vae_epochs} epochs...\n")

for epoch in range(vae_epochs):
    vae.train()
    epoch_total = 0
    epoch_recon = 0
    epoch_kl = 0
    
    for batch_idx, (images, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{vae_epochs}", leave=False)):
        images = images.to(device)
        
        # Forward pass
        recon, mu, logvar, z = vae(images)
        
        # Compute loss
        total_loss, recon_loss, kl_loss = vae_loss(recon, images, mu, logvar, beta=vae_beta)
        
        # Backpropagation
        vae_optimizer.zero_grad()
        total_loss.backward()
        vae_optimizer.step()
        
        epoch_total += total_loss.item()
        epoch_recon += recon_loss.item()
        epoch_kl += kl_loss.item()
    
    # Average losses
    avg_total = epoch_total / len(train_loader)
    avg_recon = epoch_recon / len(train_loader)
    avg_kl = epoch_kl / len(train_loader)
    
    vae_history['total'].append(avg_total)
    vae_history['recon'].append(avg_recon)
    vae_history['kl'].append(avg_kl)
    
    print(f"Epoch {epoch+1}/{vae_epochs} - Total: {avg_total:.4f}, Recon: {avg_recon:.4f}, KL: {avg_kl:.4f}")

print("\nVAE training complete!")

## Part 11: Visualizing VAE Compression Quality

Let's see how well the VAE compresses and reconstructs images. This is critical - poor VAE quality means poor final image quality!

In [ ]:
def show_images(images, title="Images", nrow=8, figsize=(12, 6)):
    """Display a grid of images (assumes images in [-1, 1] range)."""
    # Denormalize from [-1, 1] to [0, 1]
    images = images * 0.5 + 0.5
    images = torch.clamp(images, 0, 1)
    
    grid = make_grid(images, nrow=nrow, padding=2)
    
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Get test images
vae.eval()
with torch.no_grad():
    test_images, _ = next(iter(test_loader))
    test_images = test_images.to(device)
    
    # Reconstruct
    recon_images, mu, logvar, z = vae(test_images)
    
    # Show original
    show_images(test_images[:64], "Original MNIST Images")
    
    # Show reconstruction
    show_images(recon_images[:64], "VAE Reconstructions")
    
    # Show latent statistics
    print(f"\nLatent space statistics:")
    print(f"  Mean: {mu.mean():.4f} ± {mu.std():.4f}")
    print(f"  Logvar: {logvar.mean():.4f} ± {logvar.std():.4f}")
    print(f"  Latent shape: {z.shape}")
    print(f"  Compression ratio: {test_images[0].numel() / z[0].numel():.1f}×")

## Part 12: Understanding the Compression Ratio

Let's quantify how much the VAE compresses the data.

In [ ]:
# Calculate compression statistics
original_size = 1 * 28 * 28  # channels × height × width
latent_size = 4 * 7 * 7  # latent_channels × latent_height × latent_width
compression_ratio = original_size / latent_size

print("VAE Compression Analysis:")
print(f"  Original image: {original_size} elements (1×28×28)")
print(f"  Latent code: {latent_size} elements (4×7×7)")
print(f"  Compression ratio: {compression_ratio:.1f}×")
print(f"\nFor comparison:")
print(f"  Stable Diffusion: 512×512×3 → 64×64×4")
print(f"  SD compression: {(512*512*3) / (64*64*4):.1f}×")
print(f"\nThis compression is key to making diffusion models fast!")

## Part 13: Diffusion in Latent Space - Theory

Now comes the breakthrough! Instead of running diffusion on pixel space, we run it on **latent space**.

### Standard DDPM (Pixel Space)

```
x₀ (image) → Add noise → xₜ (noisy image) → U-Net predicts noise → x₀
```

Problem: x₀ is high-dimensional (28×28 = 784 for MNIST, 512×512×3 = 786,432 for HD images)

### Latent Diffusion (Latent Space)

```
x₀ (image) → VAE encode → z₀ (latent) → Add noise → zₜ (noisy latent) → U-Net → z₀ → VAE decode → x₀
```

Benefit: z₀ is much smaller (7×7×4 = 196 for our MNIST, 64×64×4 = 16,384 for Stable Diffusion)

### The Math

**Forward process in latent space:**
$$z_t = \sqrt{\bar{\alpha}_t} \cdot z_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon$$

**Training objective:**
$$\mathcal{L} = \mathbb{E}_{z_0, t, \epsilon} \left[ \| \epsilon - \epsilon_\theta(z_t, t) \|^2 \right]$$

where:
- $z_0 = \text{VAE.encode}(x_0)$ is the latent encoding
- $\epsilon_\theta$ is a U-Net that predicts noise in latent space

### Why This Works

1. **Latents are semantic**: VAE learns to capture "what" is in the image
2. **Removes redundancy**: VAE strips out high-frequency details that don't affect perception
3. **Smaller space**: U-Net operates on much smaller representations
4. **Faster**: 64× fewer elements to process for each denoising step

## Part 14: Noise Schedule for Latent Diffusion

We'll use the same noise schedule as standard diffusion, but apply it to latent vectors instead of images.

In [ ]:
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """Linear schedule for beta values."""
    return torch.linspace(beta_start, beta_end, timesteps)

# Create noise schedule
timesteps = 1000
betas = linear_beta_schedule(timesteps)

# Compute alphas and alpha_bar
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

# Compute square roots for sampling
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

# For reverse process
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

print(f"Noise schedule:")
print(f"  Timesteps: {timesteps}")
print(f"  Beta range: [{betas.min():.6f}, {betas.max():.6f}]")
print(f"  Alpha_bar at t=0: {alphas_cumprod[0]:.6f}")
print(f"  Alpha_bar at t={timesteps-1}: {alphas_cumprod[-1]:.6f}")

## Part 15: Forward Diffusion in Latent Space

Let's implement the forward process that adds noise to latent representations.

In [ ]:
def extract(a, t, x_shape):
    """Extract coefficients at specified timesteps."""
    batch_size = t.shape[0]
    # Move to CPU for gather, then back to target device
    a_cpu = a.cpu()
    t_cpu = t.cpu()
    out = a_cpu.gather(-1, t_cpu)
    out = out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))
    return out.to(t.device)

def forward_diffusion_latent(z_0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, noise=None):
    """
    Apply forward diffusion to latent vectors.
    
    Same as pixel-space diffusion, but operates on latent space!
    
    Args:
        z_0: Clean latents (batch_size, latent_channels, latent_h, latent_w)
        t: Timesteps (batch_size,)
        sqrt_alphas_cumprod: Precomputed coefficients
        sqrt_one_minus_alphas_cumprod: Precomputed coefficients
        noise: Optional noise (will be sampled if None)
    
    Returns:
        z_t: Noisy latents at timestep t
        noise: The noise that was added
    """
    if noise is None:
        noise = torch.randn_like(z_0)
    
    # Extract coefficients for timestep t
    sqrt_alpha_bar_t = extract(sqrt_alphas_cumprod, t, z_0.shape)
    sqrt_one_minus_alpha_bar_t = extract(sqrt_one_minus_alphas_cumprod, t, z_0.shape)
    
    # Apply forward diffusion formula
    z_t = sqrt_alpha_bar_t * z_0 + sqrt_one_minus_alpha_bar_t * noise
    
    return z_t, noise

# Test forward diffusion on latents
vae.eval()
with torch.no_grad():
    test_img, _ = next(iter(test_loader))
    test_img = test_img[:1].to(device)
    
    # Encode to latent
    test_z, _, _ = vae.encode(test_img)
    
    # Add noise at different timesteps
    timesteps_to_test = [0, 250, 500, 750, 999]
    
    print("Forward diffusion in latent space:")
    for t in timesteps_to_test:
        t_tensor = torch.tensor([t]).to(device)
        z_t, _ = forward_diffusion_latent(
            test_z, t_tensor,
            sqrt_alphas_cumprod.to(device),
            sqrt_one_minus_alphas_cumprod.to(device)
        )
        print(f"  t={t:4d}: z_t range [{z_t.min():.2f}, {z_t.max():.2f}]")

## Part 16: Visualizing Latent Space Diffusion

Let's visualize what happens when we add noise to latents and then decode them back to images.

In [ ]:
# Visualize diffusion in latent space
vae.eval()
with torch.no_grad():
    # Get one image
    img, label = test_dataset[0]
    img = img.unsqueeze(0).to(device)
    
    # Encode to latent
    z_0, _, _ = vae.encode(img)
    
    # Add noise at different timesteps
    timesteps_to_show = [0, 100, 250, 500, 750, 999]
    noisy_images = []
    
    # Use same noise for consistency
    noise = torch.randn_like(z_0)
    
    for t in timesteps_to_show:
        t_tensor = torch.tensor([t]).to(device)
        z_t, _ = forward_diffusion_latent(
            z_0, t_tensor,
            sqrt_alphas_cumprod.to(device),
            sqrt_one_minus_alphas_cumprod.to(device),
            noise=noise
        )
        
        # Decode noisy latent back to image
        img_t = vae.decode(z_t)
        noisy_images.append(img_t)
    
    # Visualize
    noisy_images = torch.cat(noisy_images, dim=0)
    
    fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 3))
    for idx, (ax, t) in enumerate(zip(axes, timesteps_to_show)):
        img_display = (noisy_images[idx, 0].cpu() * 0.5 + 0.5).clamp(0, 1)
        ax.imshow(img_display, cmap='gray')
        ax.set_title(f't = {t}')
        ax.axis('off')
    
    plt.suptitle('Latent Diffusion: Adding Noise in Latent Space', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("Notice: Noise is added in latent space but we decode to visualize the effect!")
    print("The U-Net will learn to denoise in this latent space.")

## Part 17: Text Conditioning - The Game Changer

### Why Text Conditioning?

So far, we can generate random samples. But what if we want to control **what** gets generated?

**Text conditioning** lets us:
- Generate "a digit 7" instead of random digits
- In Stable Diffusion: "a cat wearing a hat" instead of random images
- Control the generation process with natural language!

### How Text Conditioning Works

**The architecture:**
```
Text "digit 7" → [Text Encoder] → Text Embedding (c)
                                        ↓
Noisy Latent (z_t) → [U-Net with Cross-Attention] → Predicted Noise
                              ↑
                        Cross-attention uses text embeddings
```

### Cross-Attention Mechanism

**In self-attention** (from Transformer notebooks):
- Query (Q), Key (K), Value (V) all come from the same input
- Attention(Q, K, V) = softmax(QK^T / √d) V

**In cross-attention**:
- **Query (Q)**: From the image features (what we're denoising)
- **Key (K) and Value (V)**: From text embeddings (what we want to generate)
- The image "attends to" the text to know what to generate!

This allows text to **guide** the denoising process at each layer of the U-Net.

### For MNIST

We'll use simple conditioning:
- Input: "Generate digit 7"
- Text encoder: Simple embedding layer (digit label → embedding)
- U-Net: Cross-attention layers that use digit embeddings

This demonstrates the concept used in Stable Diffusion (which uses CLIP text encoder)!

## Part 18: Implementing Cross-Attention

Let's build a cross-attention layer that will allow text to guide image generation.

In [ ]:
class CrossAttention(nn.Module):
    """
    Cross-attention layer for conditioning.
    
    Query comes from image features.
    Key and Value come from conditioning (text embeddings).
    """
    def __init__(self, query_dim, context_dim, num_heads=4, head_dim=64):
        super().__init__()
        
        self.num_heads = num_heads
        self.head_dim = head_dim
        inner_dim = num_heads * head_dim
        
        # Query projection (from image features)
        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        
        # Key and Value projections (from text embeddings)
        self.to_k = nn.Linear(context_dim, inner_dim, bias=False)
        self.to_v = nn.Linear(context_dim, inner_dim, bias=False)
        
        # Output projection
        self.to_out = nn.Linear(inner_dim, query_dim)
    
    def forward(self, x, context):
        """
        Args:
            x: Image features (batch, seq_len, query_dim)
            context: Text embeddings (batch, context_len, context_dim)
        
        Returns:
            Output features (batch, seq_len, query_dim)
        """
        batch_size = x.shape[0]
        
        # Project to Q, K, V
        q = self.to_q(x)  # (batch, seq_len, inner_dim)
        k = self.to_k(context)  # (batch, context_len, inner_dim)
        v = self.to_v(context)  # (batch, context_len, inner_dim)
        
        # Reshape for multi-head attention
        q = q.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scale = self.head_dim ** -0.5
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * scale, dim=-1)
        
        # Apply attention to values
        out = torch.matmul(attn, v)  # (batch, num_heads, seq_len, head_dim)
        
        # Reshape back
        out = out.transpose(1, 2).reshape(batch_size, -1, self.num_heads * self.head_dim)
        
        return self.to_out(out)

# Test cross-attention
test_cross_attn = CrossAttention(query_dim=64, context_dim=128, num_heads=4, head_dim=16).to(device)
test_x = torch.randn(4, 49, 64).to(device)  # Image features (7×7 spatial)
test_context = torch.randn(4, 1, 128).to(device)  # Text embeddings
test_out = test_cross_attn(test_x, test_context)

print(f"Cross-attention test:")
print(f"  Image features: {test_x.shape}")
print(f"  Text embeddings: {test_context.shape}")
print(f"  Output: {test_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in test_cross_attn.parameters()):,}")

## Part 19: Time Embeddings for Conditioning

The U-Net needs to know which timestep it's denoising. We use sinusoidal position embeddings (same as in Transformers).

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal position embeddings for timestep encoding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# Test
time_emb = SinusoidalPositionEmbeddings(dim=128)
test_times = torch.tensor([0, 250, 500, 750, 999])
test_emb = time_emb(test_times)

print(f"Time embedding test:")
print(f"  Input: {test_times.shape}")
print(f"  Output: {test_emb.shape}")

## Part 20: Building the Conditioned U-Net

Now we build a U-Net that:
1. Takes noisy latents as input
2. Uses time embeddings to know which timestep
3. Uses cross-attention to condition on text/class labels
4. Predicts the noise to remove

In [ ]:
class ConditionedResBlock(nn.Module):
    """
    Residual block with time and text conditioning.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, context_dim=None):
        super().__init__()
        
        # First conv
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        )
        
        # Time embedding projection
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        
        # Cross-attention for text conditioning (optional)
        self.use_cross_attn = context_dim is not None
        if self.use_cross_attn:
            self.cross_attn = CrossAttention(
                query_dim=out_channels,
                context_dim=context_dim,
                num_heads=4,
                head_dim=out_channels // 4
            )
            self.norm_cross = nn.GroupNorm(8, out_channels)
        
        # Second conv
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        )
        
        # Residual connection
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    
    def forward(self, x, time_emb, context=None):
        residual = self.residual(x)
        
        # First conv
        h = self.conv1(x)
        
        # Add time embedding
        time_emb = self.time_mlp(time_emb)
        h = h + time_emb[:, :, None, None]
        
        # Cross-attention with text (if provided)
        if self.use_cross_attn and context is not None:
            # Reshape for attention: (B, C, H, W) → (B, H*W, C)
            B, C, H, W = h.shape
            h_attn = h.reshape(B, C, H*W).transpose(1, 2)
            
            # Apply cross-attention
            h_attn = self.cross_attn(h_attn, context)
            
            # Reshape back: (B, H*W, C) → (B, C, H, W)
            h_attn = h_attn.transpose(1, 2).reshape(B, C, H, W)
            h = h + self.norm_cross(h_attn)
        
        # Second conv
        h = self.conv2(h)
        
        return h + residual

# Test
test_block = ConditionedResBlock(64, 128, time_emb_dim=128, context_dim=64).to(device)
test_x = torch.randn(4, 64, 7, 7).to(device)
test_t = torch.randn(4, 128).to(device)
test_c = torch.randn(4, 1, 64).to(device)
test_out = test_block(test_x, test_t, test_c)

print(f"Conditioned ResBlock test:")
print(f"  Input: {test_x.shape}")
print(f"  Output: {test_out.shape}")

## Part 21: Simple Latent U-Net

For MNIST's small latent space (7×7), we'll use a simplified U-Net with cross-attention.

In [ ]:
class SimpleLatentUNet(nn.Module):
    """
    Simplified U-Net for latent diffusion on MNIST.
    
    Operates on 7×7×4 latent space.
    Includes cross-attention for text/class conditioning.
    """
    def __init__(self, latent_channels=4, context_dim=64, time_emb_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(64),
            nn.Linear(64, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )
        
        # Initial projection
        self.conv_in = nn.Conv2d(latent_channels, 64, kernel_size=3, padding=1)
        
        # Encoder (downsampling not needed for 7×7, just increase channels)
        self.enc1 = ConditionedResBlock(64, 128, time_emb_dim, context_dim)
        self.enc2 = ConditionedResBlock(128, 256, time_emb_dim, context_dim)
        
        # Bottleneck
        self.mid1 = ConditionedResBlock(256, 256, time_emb_dim, context_dim)
        self.mid2 = ConditionedResBlock(256, 256, time_emb_dim, context_dim)
        
        # Decoder
        self.dec1 = ConditionedResBlock(512, 256, time_emb_dim, context_dim)  # 512 = 256 (mid) + 256 (skip from enc2)
        self.dec2 = ConditionedResBlock(384, 128, time_emb_dim, context_dim)  # 384 = 256 (dec1) + 128 (skip from enc1)
        self.dec3 = ConditionedResBlock(192, 64, time_emb_dim, context_dim)   # 192 = 128 (dec2) + 64 (skip from conv_in)
        
        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, 64),
            nn.SiLU(),
            nn.Conv2d(64, latent_channels, kernel_size=3, padding=1)
        )
    
    def forward(self, z_t, t, context=None):
        """
        Args:
            z_t: Noisy latents (batch, latent_channels, 7, 7)
            t: Timesteps (batch,)
            context: Text/class embeddings (batch, 1, context_dim) or None
        
        Returns:
            Predicted noise (batch, latent_channels, 7, 7)
        """
        # Time embedding
        t_emb = self.time_embed(t)
        
        # Initial conv
        h0 = self.conv_in(z_t)
        
        # Encoder with skip connections
        h1 = self.enc1(h0, t_emb, context)
        h2 = self.enc2(h1, t_emb, context)
        
        # Bottleneck
        h = self.mid1(h2, t_emb, context)
        h = self.mid2(h, t_emb, context)
        
        # Decoder with skip connections
        h = torch.cat([h, h2], dim=1)  # Skip connection from enc2
        h = self.dec1(h, t_emb, context)
        
        h = torch.cat([h, h1], dim=1)  # Skip connection from enc1
        h = self.dec2(h, t_emb, context)
        
        h = torch.cat([h, h0], dim=1)  # Skip connection from conv_in
        h = self.dec3(h, t_emb, context)
        
        # Output
        return self.conv_out(h)

# Create model
unet = SimpleLatentUNet(latent_channels=4, context_dim=64).to(device)

# Test
test_z = torch.randn(4, 4, 7, 7).to(device)
test_t = torch.randint(0, 1000, (4,)).to(device)
test_context = torch.randn(4, 1, 64).to(device)
test_noise_pred = unet(test_z, test_t, test_context)

print(f"U-Net test:")
print(f"  Input latent: {test_z.shape}")
print(f"  Timesteps: {test_t.shape}")
print(f"  Context: {test_context.shape}")
print(f"  Predicted noise: {test_noise_pred.shape}")
print(f"  Total parameters: {sum(p.numel() for p in unet.parameters()):,}")

## Part 22: Text Encoder - Simple Class Embeddings

For MNIST, our "text" is just digit labels (0-9). We'll use a simple embedding layer. This demonstrates the concept used in Stable Diffusion (which uses CLIP embeddings).

In [ ]:
class ClassEmbedder(nn.Module):
    """
    Simple class embedding layer.
    
    Maps digit labels (0-9) to embedding vectors.
    This simulates a text encoder for our simple case.
    """
    def __init__(self, num_classes=10, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)
        # Add special token for unconditional generation
        self.null_embedding = nn.Parameter(torch.randn(1, 1, embed_dim))
    
    def forward(self, labels, use_null=None):
        """
        Args:
            labels: Class labels (batch,)
            use_null: Boolean mask for unconditional generation (batch,)
        
        Returns:
            Embeddings (batch, 1, embed_dim)
        """
        # Get class embeddings
        emb = self.embedding(labels).unsqueeze(1)  # (batch, 1, embed_dim)
        
        # Replace with null embedding for unconditional samples
        if use_null is not None:
            null_emb = self.null_embedding.expand(emb.size(0), -1, -1)
            emb = torch.where(use_null[:, None, None], null_emb, emb)
        
        return emb

# Test
text_encoder = ClassEmbedder(num_classes=10, embed_dim=64).to(device)
test_labels = torch.randint(0, 10, (8,)).to(device)
test_emb = text_encoder(test_labels)

print(f"Text encoder test:")
print(f"  Labels: {test_labels}")
print(f"  Embeddings shape: {test_emb.shape}")
print(f"  Embedding range: [{test_emb.min():.2f}, {test_emb.max():.2f}]")

## Part 23: Latent Diffusion Training Loss

The training is conceptually the same as pixel-space diffusion, but operates on latents:

1. Encode image to latent: $z_0 = \text{VAE.encode}(x)$
2. Sample timestep $t$ and add noise: $z_t = \sqrt{\bar{\alpha}_t} z_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$
3. Get text embedding: $c = \text{TextEncoder}(\text{label})$
4. Predict noise: $\hat{\epsilon} = \epsilon_\theta(z_t, t, c)$
5. Compute loss: $\mathcal{L} = \|\epsilon - \hat{\epsilon}\|^2$

In [ ]:
def latent_diffusion_loss(
    vae, unet, text_encoder,
    images, labels,
    sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod,
    timesteps,
    unconditional_prob=0.1
):
    """
    Compute latent diffusion training loss.
    
    Args:
        vae: VAE for encoding/decoding
        unet: U-Net for noise prediction
        text_encoder: Embeds class labels
        images: Input images
        labels: Class labels
        sqrt_alphas_cumprod: Diffusion coefficients
        sqrt_one_minus_alphas_cumprod: Diffusion coefficients
        timesteps: Number of diffusion steps
        unconditional_prob: Probability of unconditional training (for CFG)
    
    Returns:
        loss: MSE between true noise and predicted noise
    """
    batch_size = images.size(0)
    
    # 1. Encode images to latents (detach to not train VAE)
    with torch.no_grad():
        z_0, _, _ = vae.encode(images)
    
    # 2. Sample random timesteps
    t = torch.randint(0, timesteps, (batch_size,), device=images.device).long()
    
    # 3. Sample noise
    noise = torch.randn_like(z_0)
    
    # 4. Create noisy latents
    z_t, _ = forward_diffusion_latent(z_0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, noise=noise)
    
    # 5. Get text embeddings (randomly drop conditioning for CFG training)
    use_null = torch.rand(batch_size, device=images.device) < unconditional_prob
    context = text_encoder(labels, use_null=use_null)
    
    # 6. Predict noise
    noise_pred = unet(z_t, t, context)
    
    # 7. Compute loss
    loss = F.mse_loss(noise_pred, noise)
    
    return loss

# Test loss computation
test_images, test_labels = next(iter(train_loader))
test_images = test_images[:4].to(device)
test_labels = test_labels[:4].to(device)

test_loss = latent_diffusion_loss(
    vae, unet, text_encoder,
    test_images, test_labels,
    sqrt_alphas_cumprod.to(device),
    sqrt_one_minus_alphas_cumprod.to(device),
    timesteps
)

print(f"Latent diffusion loss test: {test_loss.item():.4f}")

## Part 24: Training the Latent Diffusion Model

Now let's train the complete latent diffusion model! The VAE is frozen - we only train the U-Net.

In [ ]:
# Training configuration
ld_epochs = 15
ld_lr = 2e-4

# Move diffusion coefficients to device
sqrt_alphas_cumprod = sqrt_alphas_cumprod.to(device)
sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod.to(device)
betas = betas.to(device)
alphas = alphas.to(device)
alphas_cumprod = alphas_cumprod.to(device)

# Freeze VAE (we only train the U-Net)
vae.eval()
for param in vae.parameters():
    param.requires_grad = False

# Optimizer for U-Net and text encoder
ld_optimizer = torch.optim.Adam(
    list(unet.parameters()) + list(text_encoder.parameters()),
    lr=ld_lr
)

# Training history
ld_history = []

print(f"Training latent diffusion for {ld_epochs} epochs...\n")
print(f"VAE is frozen. Training U-Net and text encoder.\n")

for epoch in range(ld_epochs):
    unet.train()
    text_encoder.train()
    epoch_loss = 0
    
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{ld_epochs}", leave=False)):
        images = images.to(device)
        labels = labels.to(device)
        
        # Compute loss
        loss = latent_diffusion_loss(
            vae, unet, text_encoder,
            images, labels,
            sqrt_alphas_cumprod,
            sqrt_one_minus_alphas_cumprod,
            timesteps,
            unconditional_prob=0.1  # 10% unconditional for CFG training
        )
        
        # Backpropagation
        ld_optimizer.zero_grad()
        loss.backward()
        ld_optimizer.step()
        
        epoch_loss += loss.item()
    
    # Average loss
    avg_loss = epoch_loss / len(train_loader)
    ld_history.append(avg_loss)
    
    print(f"Epoch {epoch+1}/{ld_epochs} - Loss: {avg_loss:.4f}")

print("\nLatent diffusion training complete!")

## Part 25: Sampling from Latent Diffusion - DDIM

To generate images:
1. Start with random noise in latent space
2. Iteratively denoise using the U-Net (conditioned on text)
3. Decode the final clean latent to an image using VAE decoder

In [ ]:
@torch.no_grad()
def ddim_sample_latent(
    unet, vae, text_encoder,
    labels,
    latent_shape,
    timesteps,
    alphas_cumprod,
    device,
    ddim_steps=50,
    guidance_scale=7.5
):
    """
    DDIM sampling in latent space with classifier-free guidance.
    
    Args:
        unet: Trained U-Net
        vae: Trained VAE
        text_encoder: Text/class encoder
        labels: Class labels to generate (batch_size,)
        latent_shape: Shape of latent (channels, height, width)
        timesteps: Total diffusion timesteps
        alphas_cumprod: Diffusion coefficients
        device: Device
        ddim_steps: Number of sampling steps
        guidance_scale: CFG strength (1.0 = no guidance, higher = stronger)
    
    Returns:
        Generated images
    """
    unet.eval()
    vae.eval()
    text_encoder.eval()
    
    batch_size = labels.size(0)
    
    # Get text embeddings
    context = text_encoder(labels, use_null=None)
    
    # Get unconditional embeddings for CFG
    uncond_context = text_encoder.null_embedding.expand(batch_size, -1, -1)
    
    # Create subsequence of timesteps
    skip = timesteps // ddim_steps
    seq = list(range(0, timesteps, skip))
    seq_next = [-1] + list(seq[:-1])
    
    # Start from random noise in latent space
    z = torch.randn(batch_size, *latent_shape, device=device)
    
    # Reverse diffusion
    for i, (t, t_next) in enumerate(tqdm(zip(reversed(seq), reversed(seq_next)), total=len(seq), desc="Sampling")):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        
        # Predict noise with conditioning
        noise_cond = unet(z, t_batch, context)
        
        if guidance_scale > 1.0:
            # Predict noise without conditioning
            noise_uncond = unet(z, t_batch, uncond_context)
            
            # Classifier-free guidance
            noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        else:
            noise_pred = noise_cond
        
        # Get alpha values
        alpha_bar_t = alphas_cumprod[t]
        
        # Predict x_0 (clean latent)
        pred_z0 = (z - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
        pred_z0 = torch.clamp(pred_z0, -5, 5)  # Clamp for stability
        
        if t_next >= 0:
            alpha_bar_t_next = alphas_cumprod[t_next]
            # DDIM update
            z = torch.sqrt(alpha_bar_t_next) * pred_z0 + torch.sqrt(1 - alpha_bar_t_next) * noise_pred
        else:
            z = pred_z0
    
    # Decode latents to images
    images = vae.decode(z)
    
    return images

print("DDIM sampling function defined")

## Part 26: Text-to-Image Generation!

Let's generate images conditioned on specific digit labels. This is the latent diffusion equivalent of "text-to-image"!

In [ ]:
# Generate one sample of each digit (0-9)
labels_to_generate = torch.arange(0, 10).to(device)

# Generate with strong guidance
generated_images = ddim_sample_latent(
    unet, vae, text_encoder,
    labels=labels_to_generate,
    latent_shape=(4, 7, 7),
    timesteps=timesteps,
    alphas_cumprod=alphas_cumprod,
    device=device,
    ddim_steps=50,
    guidance_scale=7.5
)

# Visualize
show_images(generated_images, "Generated Digits (0-9) with Latent Diffusion", nrow=10, figsize=(15, 2))

print(f"\nGenerated digits: {labels_to_generate.cpu().numpy()}")
print(f"This is text-to-image generation for MNIST!")

## Part 27: Classifier-Free Guidance - Theory

### What is Classifier-Free Guidance (CFG)?

**The problem:** Text conditioning helps guide generation, but sometimes the guidance is weak.

**The solution:** Amplify the effect of conditioning by:
1. Predicting noise **with** conditioning: $\epsilon_{\text{cond}} = \epsilon_\theta(z_t, t, c)$
2. Predicting noise **without** conditioning: $\epsilon_{\text{uncond}} = \epsilon_\theta(z_t, t, \emptyset)$
3. Extrapolate in the direction of conditioning:

$$\hat{\epsilon} = \epsilon_{\text{uncond}} + w \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

where $w$ is the **guidance scale**.

### Intuition

- $w = 1.0$: Normal conditional generation
- $w > 1.0$: Stronger conditioning ("push harder toward the text")
- $w < 1.0$: Weaker conditioning
- $w = 0.0$: Unconditional generation

### Training for CFG

We need the model to handle **both** conditional and unconditional inputs:
- During training, randomly drop conditioning with probability $p$ (e.g., 10%)
- When dropped, use a null/empty embedding
- This trains the model to work with and without conditioning

### Why This Works

CFG "exaggerates" the difference between conditional and unconditional predictions:
- If text says "generate a 7", CFG pushes even harder toward "7-ness"
- Results in sharper, more aligned generations
- Trade-off: Higher $w$ can reduce diversity

**Typical values:**
- Stable Diffusion-style systems: $w \approx 5-10$
- Our MNIST model: $w = 3-10$

## Part 28: Comparing Different Guidance Scales

Let's generate the same digit with different guidance scales to see the effect.

In [ ]:
# Generate digit "7" with different guidance scales
target_digit = 7
guidance_scales = [1.0, 3.0, 5.0, 7.5, 10.0]

all_samples = []

for guidance in guidance_scales:
    labels = torch.full((8,), target_digit).to(device)
    
    samples = ddim_sample_latent(
        unet, vae, text_encoder,
        labels=labels,
        latent_shape=(4, 7, 7),
        timesteps=timesteps,
        alphas_cumprod=alphas_cumprod,
        device=device,
        ddim_steps=50,
        guidance_scale=guidance
    )
    
    all_samples.append(samples)

# Visualize all
fig, axes = plt.subplots(len(guidance_scales), 8, figsize=(12, 8))

for i, (samples, guidance) in enumerate(zip(all_samples, guidance_scales)):
    for j in range(8):
        img = (samples[j, 0].cpu() * 0.5 + 0.5).clamp(0, 1)
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(f'w={guidance}', rotation=0, ha='right', va='center', fontsize=10)

plt.suptitle(f'Classifier-Free Guidance Effect (Digit {target_digit})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("  w=1.0: Diverse but may not match prompt well")
print("  w=3-5: Good balance of quality and diversity")
print("  w=7.5: Strong alignment with prompt")
print("  w=10+: Very strong alignment but less diversity")

## Part 29: Stable Diffusion Architecture - Complete Picture

Now that we've built latent diffusion, let's understand how **Stable Diffusion** works!

### Stable Diffusion Components

**1. VAE (Variational Autoencoder)**
- Encoder: 512×512×3 RGB image → 64×64×4 latent
- 8× spatial compression (64× total reduction)
- Decoder: 64×64×4 latent → 512×512×3 RGB image
- Trained separately on large image datasets

**2. CLIP Text Encoder**
- Converts text prompt into embeddings
- Uses OpenAI's CLIP model (pretrained)
- Output: 77 tokens × 768-dimensional embeddings
- Frozen during diffusion training

**3. U-Net with Cross-Attention**
- Operates on 64×64×4 latent space
- Multiple ResNet blocks with cross-attention
- Cross-attention layers use CLIP embeddings
- Time embeddings at each layer
- ~860M parameters

**4. Diffusion Scheduler**
- DDPM, DDIM, or other samplers
- Typically 50-100 steps for DDIM
- 1000 steps for DDPM training

### The Full Pipeline

```
Training:
1. Image → VAE Encoder → Latent z₀
2. Text → CLIP → Text embeddings c
3. Add noise: zₜ = √(α̅ₜ)z₀ + √(1-α̅ₜ)ε
4. U-Net predicts noise: ε̂ = U-Net(zₜ, t, c)
5. Loss: ||ε - ε̂||²

Sampling:
1. Text → CLIP → c
2. Random noise z_T ~ N(0, I)
3. For t = T..1:
     - Predict noise: ε = U-Net(zₜ, t, c)
     - Apply CFG if desired
     - Compute zₜ₋₁
4. Clean latent z₀ → VAE Decoder → Image
```

### Why Latent Diffusion is Powerful

**1. Efficiency:**
- 64× fewer elements to process
- Can run on consumer GPUs (8-16GB VRAM)
- Fast iteration during research

**2. Quality:**
- Perceptual compression preserves important details
- Semantic latent space is easier to model
- Better than pixel-space diffusion at similar compute

**3. Flexibility:**
- Easy to add conditioning (text, class, image, etc.)
- Can fine-tune on specific domains
- LoRA and other efficient fine-tuning methods
- Inpainting, img2img, controlnet all build on this

**4. Open Source:**
- Stable Diffusion weights are public
- Can run locally, modify, and extend
- Huge community and ecosystem

## Part 30: Applications and Extensions

### Image-to-Image (img2img)

Instead of starting from pure noise, start from a noisy version of an existing image:

```python
# Instead of: z_T ~ N(0, I)
z_0 = vae.encode(source_image)
z_t = add_noise(z_0, t=500)  # Partial noise
# Then denoise with new prompt
```

**Use cases:**
- Style transfer ("make this photo look like a painting")
- Image variations ("same subject, different pose")
- Upscaling/enhancement

### Inpainting

Fill in masked regions while preserving unmasked areas:

```python
# During denoising:
z_denoised = unet(z_t, t, context)
z_t_minus_1 = denoise_step(z_denoised)

# Replace unmasked regions with original (noisy) content
z_t_minus_1 = mask * z_t_minus_1 + (1 - mask) * z_original_t_minus_1
```

**Use cases:**
- Remove objects from images
- Extend images (outpainting)
- Fix/replace parts of images

### ControlNet

Add structural control (edges, pose, depth) to generation:

```python
# Extract structure (e.g., edges with Canny)
edges = canny_detector(image)

# ControlNet processes structure
control_features = controlnet(edges, t)

# Inject into U-Net
unet_output = unet(z_t, t, text_emb, control=control_features)
```

**Use cases:**
- Pose-guided generation ("person in this exact pose")
- Depth-guided generation (3D control)
- Scribble-to-image
- Semantic segmentation → realistic image

### LoRA (Low-Rank Adaptation)

Efficient fine-tuning by learning small "adapter" layers:

```python
# Instead of fine-tuning full U-Net:
# Add low-rank matrices A, B to each layer
W_new = W_pretrained + A @ B  # where A, B are small

# Only train A, B (much fewer parameters)
```

**Use cases:**
- Fine-tune on specific art styles ("Ghibli style")
- Personalization ("images of me")
- Domain adaptation
- Multiple LoRAs can be combined!

### Dreambooth

Teach the model a new concept with just 3-5 images:

```python
# Fine-tune on small dataset of subject
# Use rare token identifier: "[V]" or "sks"
prompt = "photo of [V] person in Paris"
```

**Use cases:**
- Generate images of specific people/pets/objects
- Product photography
- Artistic style learning

## Part 31: Key Takeaways and Summary

### What We Learned

#### 1. The Latent Diffusion Breakthrough
- **Problem**: Pixel-space diffusion is slow (512×512 = 262k pixels)
- **Solution**: Diffuse in compressed latent space (64×64 = 4k latents)
- **Result**: 64× fewer elements, much faster training and sampling

#### 2. VAE for Perceptual Compression
- Encoder compresses images to semantic latents
- Decoder reconstructs high-quality images
- Removes perceptually unimportant details
- Trained separately from diffusion model

#### 3. Diffusion in Latent Space
- Same forward/reverse process as pixel diffusion
- But operates on compressed latents!
- U-Net predicts noise in latent space
- Final latent is decoded to image

#### 4. Text Conditioning via Cross-Attention
- Text encoder converts prompts to embeddings
- Cross-attention layers in U-Net use text to guide denoising
- Query from image, Key/Value from text
- Enables controllable generation!

#### 5. Classifier-Free Guidance
- Amplifies effect of conditioning
- Formula: $\hat{\epsilon} = \epsilon_{uncond} + w \cdot (\epsilon_{cond} - \epsilon_{uncond})$
- Higher $w$ = stronger adherence to prompt
- Requires training with random conditioning dropout

#### 6. DDIM Sampling
- Deterministic, fast sampling
- 50-100 steps instead of 1000
- Same quality as DDPM with 10-20× speedup

### Stable Diffusion Architecture

**Components:**
1. **VAE**: 512×512×3 → 64×64×4 compression
2. **CLIP Text Encoder**: Text → embeddings
3. **U-Net with Cross-Attention**: Denoises latents, conditioned on text
4. **DDIM/DDPM Scheduler**: Controls sampling process

**Pipeline:**
- Training: Image → VAE → Latent → Add noise → U-Net predicts noise
- Sampling: Random noise → Iterative denoising (guided by text) → VAE decode → Image

### Why Latent Diffusion Matters

**Before latent diffusion:**
- High-res generation required massive compute
- Research labs only
- Slow iteration

**After latent diffusion:**
- Runs on consumer GPUs (RTX 3090, 4090)
- Open source models (Stable Diffusion)
- Fast iteration and fine-tuning
- Huge community and applications

### Real-World Impact

**Representative latent diffusion families:**
- **Stable Diffusion**: Open-source text-to-image
- **ControlNet**: Structured conditioning on top of Stable Diffusion
- Other open-source Stable Diffusion derivatives for editing, inpainting, and personalization

**Applications:**
- Art generation and exploration
- Product design and visualization
- Image editing and inpainting
- Style transfer and variation
- Video generation (AnimateDiff, etc.)

### Extensions and Future Directions

**Current extensions:**
- **ControlNet**: Structure-guided generation
- **LoRA**: Efficient fine-tuning
- **Dreambooth**: Personalization
- **Img2img**: Image-to-image translation
- **Inpainting**: Masked region filling

**Future directions:**
- Faster sampling (1-4 steps)
- Higher resolution (SDXL: 1024×1024)
- Video diffusion models
- 3D generation
- Multi-modal conditioning

### Comparison with Other Approaches

| Aspect | VAE | GAN | Pixel Diffusion | Latent Diffusion |
|--------|-----|-----|-----------------|------------------|
| **Sample Quality** | Blurry | Sharp | Sharp | Sharp |
| **Training Stability** | ✅ Stable | ❌ Difficult | ✅ Stable | ✅ Stable |
| **Sampling Speed** | ✅ Fast | ✅ Fast | ❌ Very slow | ⚠️ Moderate |
| **Controllability** | ⚠️ Limited | ⚠️ Limited | ✅ Good | ✅ Excellent |
| **Resolution** | ⚠️ Limited | ✅ High | ❌ Low | ✅ High |
| **Compute** | ✅ Low | ✅ Low | ❌ Very high | ⚠️ Moderate |

**Latent diffusion is the current state-of-the-art for controllable high-quality generation!**

### Final Thoughts

Latent diffusion represents a **paradigm shift** in generative modeling:

1. **Efficiency**: VAE compression makes high-res generation feasible
2. **Quality**: Matches or exceeds GANs without training instability
3. **Control**: Text conditioning via cross-attention is intuitive and powerful
4. **Flexibility**: Easy to extend (ControlNet, LoRA, inpainting, etc.)
5. **Accessibility**: Open source models democratize AI art

Understanding latent diffusion is **essential** for:
- Modern generative AI development
- Fine-tuning and customizing models
- Building applications on top of Stable Diffusion
- Research in generative modeling

This is the foundation of the current AI art revolution!

## Reflection Questions

1. **Efficiency**: Why is latent diffusion so much faster than pixel-space diffusion? What's the compression ratio for Stable Diffusion?

2. **Architecture**: Explain the role of each component: VAE, U-Net, text encoder, scheduler. Why is each necessary?

3. **Cross-Attention**: How does cross-attention differ from self-attention? Why is it perfect for text conditioning?

4. **Classifier-Free Guidance**: Explain CFG in your own words. What happens when guidance scale is 1.0 vs 10.0?

5. **Trade-offs**: What are the trade-offs of latent diffusion vs pixel diffusion? When might you use each?

6. **VAE Quality**: Why is VAE quality critical for latent diffusion? What happens if the VAE is poor?

7. **Applications**: How would you modify latent diffusion for: (a) image editing, (b) style transfer, (c) super-resolution?

8. **Stable Diffusion**: How does our MNIST implementation relate to Stable Diffusion? What would you need to change for real images?

Take time to deeply understand these concepts - latent diffusion is the foundation of modern generative AI!